© 2026 Mechanome Bio. Proprietary and confidential.

# Run RNA2seg
repo: https://github.com/fish-quant/rna2seg , docs: https://rna2seg.readthedocs.io/en/latest/

- Package `rna2seg` (0.1.5), Python >=3.10. Own env (`rna2seg`, python 3.11), built in the first cell.
- Works directly on a spatialdata zarr. The crop is written to `../inputs/RNA2seg/<dataset>_crop.zarr` because
  RNA2seg writes its intermediate elements (patches, consistent cells, results) into the zarr it's given.
- Pretrained weights are from other tissues, so the model is trained on our data. Training targets come from
  `compute_consistent_cell`, i.e. 10x cells/nuclei that agree with each other (minimal input, no ground truths).
- Workflow follows the docs notebooks `train_model_on_zarr.ipynb` and `apply_model_on_zarr.ipynb`.

## Environment (run once, from any kernel; then switch this notebook's kernel to `rna2seg`)
rna2seg is installed with `--no-deps`: it pins sopa==2.0.4, which can't read zarr v3. Its dependencies are in
`env_files/rna2seg.txt` with sopa 2.1.6 and instanseg-torch 0.0.9 instead.

In [ ]:
!conda create -y -n rna2seg python=3.11 pip
!conda run --no-capture-output -n rna2seg pip install -r ../env_files/rna2seg.txt
!conda run --no-capture-output -n rna2seg pip install --no-deps rna2seg==0.1.5
!conda run -n rna2seg python -m ipykernel install --user --name rna2seg --display-name "rna2seg"

In [ ]:
import sys
import warnings

import albumentations
import cv2
import numpy
import spatialdata
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from tqdm import tqdm

sys.path.append("..")
from utils import seg_utils

warnings.filterwarnings("ignore")

DATASET_NAME = "20240208_core_s0005_spt-xen_onco_pancreas"
METHOD = "rna2seg"
BOUNDING_BOX_MICRONS = (4673, 1476, 5036, 1805)  # microns; None = whole slide

WORK_DIRECTORY = seg_utils.INPUTS_DIRECTORY / "RNA2seg"
CROP_ZARR = WORK_DIRECTORY / f"{DATASET_NAME}_crop.zarr"
MODEL_DIRECTORY = WORK_DIRECTORY / f"{DATASET_NAME}_model"
WORK_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

# patch sizes from the docs: 1200/50 for training, 1200/150 for inference
PATCH_WIDTH = 1200
TRAINING_PATCH_OVERLAP = 50
INFERENCE_PATCH_OVERLAP = 150
NUMBER_OF_EPOCHS = 3  # as in the docs notebook

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sdata = seg_utils.load_sdata(DATASET_NAME)

In [ ]:
seg_utils.preview_bounding_box(sdata, BOUNDING_BOX_MICRONS)

In [ ]:
sdata_crop = seg_utils.crop_sdata(sdata, BOUNDING_BOX_MICRONS)
sdata_crop = sdata_crop.subset([seg_utils.IMAGE_KEY, seg_utils.TRANSCRIPTS_KEY, seg_utils.NUCLEI_KEY, seg_utils.CELLS_KEY])
sdata_crop.write(CROP_ZARR, overwrite=True)
sdata_work = spatialdata.read_zarr(CROP_ZARR)

## Training targets
Patches + per-patch transcript csvs, then the consistent cell/nucleus shapes.

In [ ]:
from rna2seg.dataset_zarr import RNA2segDataset, compute_consistent_cell, create_patch_rna2seg, custom_collate_fn

training_patch_directory = CROP_ZARR / f".rna2seg_{PATCH_WIDTH}_{TRAINING_PATCH_OVERLAP}"
create_patch_rna2seg(
    sdata=sdata_work, image_key=seg_utils.IMAGE_KEY, points_key=seg_utils.TRANSCRIPTS_KEY,
    patch_width=PATCH_WIDTH, patch_overlap=TRAINING_PATCH_OVERLAP, min_points_per_patch=0,
    folder_patch_rna2seg=training_patch_directory, gene_column_name=seg_utils.GENE_COLUMN, overwrite=True,
)

sdata_work, _ = compute_consistent_cell(
    sdata=sdata_work,
    key_shape_nuclei_seg=seg_utils.NUCLEI_KEY,
    key_shape_cell_seg=seg_utils.CELLS_KEY,
    key_cell_consistent="cell_consistent",
    key_nuclei_consistent="nucleus_consistent",
    image_key=seg_utils.IMAGE_KEY,
    threshold_intersection_contain=0.95,
    threshold_intersection_intersect=0.05,
    accepted_nb_nuclei_per_cell=None,
    max_cell_nb_intersecting_nuclei=1,
)

## Train

In [ ]:
from rna2seg.models import RNA2seg
from rna2seg.train import train_one_epoch

resize_to_512 = albumentations.Compose([albumentations.Resize(width=512, height=512, interpolation=cv2.INTER_NEAREST)])

training_dataset = RNA2segDataset(
    sdata=sdata_work,
    channels_dapi=[seg_utils.DAPI_CHANNEL],
    channels_cellbound=seg_utils.BOUNDARY_CHANNELS,
    shape_patch_key=f"sopa_patches_rna2seg_{PATCH_WIDTH}_{TRAINING_PATCH_OVERLAP}",
    key_cell_consistent="cell_consistent",
    key_nucleus_consistent="nucleus_consistent",
    key_nuclei_segmentation=seg_utils.NUCLEI_KEY,
    gene_column=seg_utils.GENE_COLUMN,
    density_threshold=None,
    kernel_size_background_density=10,
    kernel_size_rna2img=0.5,
    max_filter_size_rna2img=2,
    transform_resize=resize_to_512,
    training_mode=True,
    patch_dir=training_patch_directory,
    patch_width=PATCH_WIDTH,
    patch_overlap=TRAINING_PATCH_OVERLAP,
    use_cache=True,
)

train_indices, validation_indices = train_test_split(range(len(training_dataset)), test_size=0.1, shuffle=True, random_state=42)
training_loader = DataLoader(training_dataset, batch_size=2, num_workers=0, sampler=SubsetRandomSampler(train_indices),
                             collate_fn=custom_collate_fn)
validation_loader = DataLoader(training_dataset, batch_size=2, num_workers=0, sampler=SubsetRandomSampler(validation_indices),
                               collate_fn=custom_collate_fn)

model = RNA2seg(device, net="unet", flow_threshold=0.9, cellbound_flow_threshold=0.4, pretrained_model=None).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

for epoch_index in tqdm(range(NUMBER_OF_EPOCHS)):
    train_one_epoch(
        device=device, epoch_index=epoch_index, rna2seg=model,
        training_loader=training_loader, optimizer=optimizer,
        print_loss_every=max(1, len(training_loader) // 3), tb_writer=None,
        validation_loader=validation_loader, path_save_model=MODEL_DIRECTORY,
        cellbound_prob=0.8, best_val_loss=numpy.inf,
    )

## Inference with the trained model (`best.pt`)

In [ ]:
from rna2seg.utils import save_shapes2zarr

inference_patch_directory = CROP_ZARR / f".rna2seg_{PATCH_WIDTH}_{INFERENCE_PATCH_OVERLAP}"
create_patch_rna2seg(
    sdata=sdata_work, image_key=seg_utils.IMAGE_KEY, points_key=seg_utils.TRANSCRIPTS_KEY,
    patch_width=PATCH_WIDTH, patch_overlap=INFERENCE_PATCH_OVERLAP, min_points_per_patch=0,
    folder_patch_rna2seg=inference_patch_directory, gene_column_name=seg_utils.GENE_COLUMN, overwrite=True,
)

inference_dataset = RNA2segDataset(
    sdata=sdata_work,
    channels_dapi=[seg_utils.DAPI_CHANNEL],
    channels_cellbound=seg_utils.BOUNDARY_CHANNELS,
    patch_width=PATCH_WIDTH,
    patch_overlap=INFERENCE_PATCH_OVERLAP,
    gene_column=seg_utils.GENE_COLUMN,
    transform_resize=resize_to_512,
    patch_dir=inference_patch_directory,
)

trained_model = RNA2seg(device, net="unet", flow_threshold=0.9, cellbound_flow_threshold=0.4,
                        pretrained_model=str(MODEL_DIRECTORY / "best.pt"))
for index in tqdm(range(len(inference_dataset))):
    trained_model.run(path_temp_save=inference_patch_directory, input_dict=inference_dataset[index])

# stitches the per-patch parquets, resolving cells in overlaps
save_shapes2zarr(dataset=inference_dataset, path_parquet_files=inference_patch_directory,
                 segmentation_key="rna2seg_cells", overwrite=True)

## Save
RNA2seg's polygons are in pixels of the cropped image, so they're shifted by the crop origin and converted to microns.

In [ ]:
sdata_work = spatialdata.read_zarr(CROP_ZARR)
polygons = seg_utils.pixel_polygons_to_microns(sdata_work["rna2seg_cells"],
                                               origin_pixels=seg_utils.image_origin_pixels(sdata_work))
seg_utils.save_segmentation(sdata, polygons, METHOD, DATASET_NAME, BOUNDING_BOX_MICRONS)